# A1 — Random Top-8000 from the original Reddit corpus

**Only A1.** No Full rebuild, no keyword/HNSW retrieval, no GPT calls. For each topic, select **8,000 distinct raw rows** uniformly at random from the same eligible Reddit corpus. Independent, reproducible samples per topic. Outputs keep the expected Step-1 schema for later GPT stance filtering.

Run `TEST` first (real random selection, 1,000 records per topic), inspect the summary, then change `RUN_MODE` to `FULL`. All checkpoints are persisted to Google Drive, including **inside each topic** every 500 selected rows. TEST and FULL use separate directories. This notebook never edits baseline files.

In [ ]:
!pip -q install pandas numpy

In [ ]:
from pathlib import Path
import json, hashlib, os
import numpy as np
import pandas as pd
from IPython.display import display

## Configuration — change paths only if needed

In [ ]:
RAW_DATA_PATH = Path('/content/Reddit/cmv_comments.csv')
OUTPUT_ROOT = Path('/content/A1_random_only_v1')
RUN_MODE = 'FULL'    # TEST first; then FULL
TOPICS = ['blm', 'guns', 'samesex']
SEED = 42
N_PER_TOPIC = 1000 if RUN_MODE == 'TEST' else 8000
CHECKPOINT_EVERY = 500
TEXT_COL = 'comment_text'
TITLE_COL = 'topic_title'
RUN_DIR = OUTPUT_ROOT / RUN_MODE.lower()
RUN_DIR.mkdir(parents=True, exist_ok=True)
print('Mode:', RUN_MODE, '| Records per topic:', N_PER_TOPIC, '| Output:', RUN_DIR)

## Load source and lock the exact eligible population

This matches the earlier ablation Step-1 eligible-row definition: `_search_text = topic_title + comment_text`; exclude only rows where both are empty. No topic-based filtering is applied. We do not rebuild any embeddings or HNSW indices.

In [ ]:
assert RUN_MODE in {'TEST', 'FULL'}
assert RAW_DATA_PATH.is_file(), RAW_DATA_PATH

def atomic_csv(df, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    df.to_csv(temp, index=False, encoding='utf-8-sig')
    os.replace(temp, path)

def atomic_json(obj, path):
    path = Path(path); path.parent.mkdir(parents=True, exist_ok=True)
    temp = path.with_name(path.name + '.tmp')
    temp.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding='utf-8')
    os.replace(temp, path)

def sha256_file(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

raw_sha = sha256_file(RAW_DATA_PATH)
raw_df = pd.read_csv(RAW_DATA_PATH, encoding='utf-8-sig', low_memory=False)
assert TEXT_COL in raw_df.columns
if 'comment_id' in raw_df.columns:
    raw_df['row_uid'] = raw_df['comment_id'].astype(str)
else:
    raw_df['row_uid'] = np.arange(len(raw_df)).astype(str)
raw_df[TEXT_COL] = raw_df[TEXT_COL].fillna('').astype(str)
titles = raw_df[TITLE_COL].fillna('').astype(str).str.strip() if TITLE_COL in raw_df else pd.Series('', index=raw_df.index)
raw_df['_search_text'] = (titles + ' ' + raw_df[TEXT_COL].str.strip()).str.strip()
raw_df = raw_df.loc[raw_df['_search_text'].str.len() > 0].reset_index(drop=True)


In [ ]:
# Remove deleted/removed comment bodies BEFORE random sampling.
# This is content hygiene, NOT topic-aware filtering.
removed_mask = raw_df[TEXT_COL].str.strip().str.casefold().isin({'[deleted]', '[removed]'})
removed_counts = raw_df.loc[removed_mask, TEXT_COL].str.strip().str.casefold().value_counts().to_dict()
print('Removed placeholder comments:', int(removed_mask.sum()), removed_counts)
raw_df = raw_df.loc[~removed_mask].reset_index(drop=True)
assert len(raw_df) >= N_PER_TOPIC, 'Not enough eligible rows after cleaning'
print('Eligible source rows after cleaning:', len(raw_df))


In [ ]:
manifest = {'raw_sha256': raw_sha, 'usable_rows': len(raw_df), 'seed': SEED,
            'topics': TOPICS, 'n_per_topic': N_PER_TOPIC, 'checkpoint_every': CHECKPOINT_EVERY, 'removed_placeholder_counts': removed_counts,
            'selection': 'uniform_without_replacement_per_topic_excluding_deleted_removed_v2'}
manifest_path = RUN_DIR / 'manifest.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text(encoding='utf-8')) == manifest, 'Existing run settings/source changed. Choose a new OUTPUT_ROOT.'
else:
    atomic_json(manifest, manifest_path)
print('Usable source rows:', len(raw_df), '| Source SHA256:', raw_sha)
display(raw_df[[TEXT_COL] + ([TITLE_COL] if TITLE_COL in raw_df else [])].head(3))

## Sample with durable intra-topic checkpoints

Each topic has a deterministic RNG seed. The entire list of selected indices is persisted **before** writing batches; every 500 rows, a cumulative checkpoint is saved. On reconnect, the notebook resumes from the last valid checkpoint, without resampling or changing order. Files are written atomically.

In [ ]:
def generate_topic(topic):
    assert topic in TOPICS
    folder = RUN_DIR / 'checkpoints' / topic
    folder.mkdir(parents=True, exist_ok=True)
    index_path = folder / 'selected_indices.npy'
    meta_path = folder / 'indices_meta.json'
    seed = SEED + TOPICS.index(topic)
    if index_path.exists():
        assert meta_path.exists(), 'Missing index metadata: ' + str(meta_path)
        meta = json.loads(meta_path.read_text(encoding='utf-8'))
        assert meta == {'topic':topic,'seed':seed,'n':N_PER_TOPIC,'source_sha256':raw_sha,'usable_rows':len(raw_df)}
        indices = np.load(index_path, allow_pickle=False)
        print(f'{topic}: loaded persisted random selection')
    else:
        indices = np.random.default_rng(seed).choice(len(raw_df), size=N_PER_TOPIC, replace=False)
        temp = folder / 'selected_indices.tmp.npy'
        np.save(temp, indices)
        os.replace(temp, index_path)
        atomic_json({'topic':topic,'seed':seed,'n':N_PER_TOPIC,'source_sha256':raw_sha,'usable_rows':len(raw_df)},meta_path)
    assert len(indices) == N_PER_TOPIC and len(np.unique(indices)) == N_PER_TOPIC
    assert indices.min() >= 0 and indices.max() < len(raw_df)
    final_path = RUN_DIR / 'top8000_variants' / f'{topic}_top8000_A1_no_topic_retrieval.csv'
    if final_path.exists():
        finished = pd.read_csv(final_path, encoding='utf-8-sig', low_memory=False)
        assert len(finished)==N_PER_TOPIC and finished['raw_index'].tolist()==indices.tolist(), 'Final output differs from saved indices'
        print(f'{topic}: final file already exists, skipping')
        return finished
    checkpoints = sorted(folder.glob('progress_*.csv'))
    if checkpoints:
        checkpoint = checkpoints[-1]
        previous = pd.read_csv(checkpoint, encoding='utf-8-sig', low_memory=False)
        assert len(previous) <= N_PER_TOPIC and previous['raw_index'].tolist() == indices[:len(previous)].tolist(), 'Checkpoint differs from selected indices'
        completed = len(previous)
        print(f'{topic}: resuming at {completed}/{N_PER_TOPIC}')
    else:
        completed = 0
    for end in range(completed + CHECKPOINT_EVERY, N_PER_TOPIC + CHECKPOINT_EVERY, CHECKPOINT_EVERY):
        end = min(end, N_PER_TOPIC)
        if end <= completed: continue
        # Reconstruct the cumulative checkpoint from the original source, avoiding CSV type drift.
        selected = raw_df.iloc[indices[:end]].copy()
        selected['raw_index'] = indices[:end]
        selected['topic'] = topic
        selected['keyword_match'] = 0
        selected['keyword_count'] = 0
        selected['hnsw_match'] = 0
        selected['hnsw_similarity'] = 0.0
        selected['candidate_source'] = 'random_raw'
        selected['topic_candidate_score'] = 0.0
        selected['top8000_variant'] = 'A1_no_topic_retrieval'
        selected['top8000_selection_mode'] = 'random_without_topic_ranking'
        atomic_csv(selected, folder / f'progress_{end:05d}.csv')
        print(f'{topic}: saved {end}/{N_PER_TOPIC}')
        completed = end
        if end == N_PER_TOPIC:
            atomic_csv(selected, final_path)
            print('FINAL:', final_path)
            return selected
    raise RuntimeError('Did not complete: '+topic)

results = {}
for topic in TOPICS:
    results[topic] = generate_topic(topic)
print('All requested A1 samples completed.')

## Real-results inspection — stop here in TEST

In [ ]:
rows=[]
for topic, df in results.items():
    rows.append({'topic':topic, 'sampled_rows':len(df),
                 'unique_raw_rows':df['raw_index'].nunique(),
                 'unique_comment_ids':df['comment_id'].nunique() if 'comment_id' in df else None,
                 'empty_comment_text':int(df['comment_text'].fillna('').astype(str).str.strip().eq('').sum()),
                 'source':'entire eligible Reddit corpus (no topic filtering)'})
    assert len(df)==N_PER_TOPIC and df['raw_index'].is_unique
    print('\n',topic, '— random examples:')
    display(df[['comment_id','comment_text','topic_title'] if 'topic_title' in df else ['comment_id','comment_text']].head(5))
summary = pd.DataFrame(rows)
display(summary)
atomic_csv(summary, RUN_DIR / 'A1_step1_summary.csv')
print('TEST complete: inspect samples and summary. Change RUN_MODE to FULL only when satisfied.')

## What comes next

In `FULL`, the three final CSVs are in `.../full/top8000_variants/` and contain 8,000 distinct **source rows** each. The three random draws are independent; overlap **between topics is allowed** because each topic is sampled independently from the same corpus. A repeated `comment_id` within a topic can still occur if the original corpus has duplicate IDs; sampling uniqueness is based on source-row indices.

These files are **unlabeled**, as required before GPT relevance/stance filtering. The next notebook must follow the original baseline Step 2, with A1 as the only altered stage. Historical A1 repository counts (202/87/53) are **not guaranteed** to recur after new sampling or GPT runs. The original Full baseline files remain untouched.